# 🕌 QuranASR v2 — Training Notebook (Run All)

**Runtime → A100 GPU + High RAM → Runtime → Run all**

That's it. This notebook does everything:
1. Installs packages
2. Mounts Drive and loads your prepared dataset
3. Trains whisper-small on Quran recitations
4. Tests on 5 validation samples
5. Pushes to HuggingFace Hub

**Prerequisites:** Run `notebook.ipynb` Cells 1–6 first to download audio and prepare the dataset.

📖 [GitHub](https://github.com/tialkan/QuranASR) · 🤗 [HuggingFace](https://huggingface.co/baristiran) · 📱 [Tilavet App](https://tilavet.app)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 1. Install packages (no restart needed — fresh kernel on Run All)
# ══════════════════════════════════════════════════════════════════════════════
import subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers', 'datasets', 'accelerate', 'evaluate',
    'jiwer', 'soundfile', 'librosa', 'huggingface_hub'],
    check=True, capture_output=True)

subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg'],
    capture_output=True)

import transformers
print(f'transformers={transformers.__version__}')
print('✅ Packages ready')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 2. Mount Drive + constants + verify data
# ══════════════════════════════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

import os, torch
from pathlib import Path

BASE        = '/content/drive/MyDrive/quran_asr'
MODEL_ID    = 'openai/whisper-small'
OUTPUT_DIR  = f'{BASE}/checkpoints/quran-whisper-small'
DATASET_DIR = f'{BASE}/data/hf_dataset'
HF_REPO_ID  = 'baristiran/QuranASR'

assert os.path.exists(f'{DATASET_DIR}/train'), f'Dataset not found! Run notebook.ipynb Cells 1-6 first.'
assert torch.cuda.is_available(), 'No GPU! Change runtime to A100 GPU + High RAM.'

print(f'GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1e9:.0f}GB)')
print('✅ Drive mounted, dataset found')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 3. Train
# ══════════════════════════════════════════════════════════════════════════════
import torch, evaluate, glob, threading, time, gc
import soundfile as sf, numpy as np
from dataclasses import dataclass, field
from typing import Any, List, Dict
from datasets import load_from_disk
import transformers
from transformers import (
    WhisperForConditionalGeneration, WhisperProcessor,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    EarlyStoppingCallback, TrainerCallback,
)

# Keep Colab alive
def _alive():
    while True:
        time.sleep(55)
        try:
            from google.colab import output; output.eval_js('void(0)')
        except: pass
threading.Thread(target=_alive, daemon=True).start()

# Dataset
train_ds = load_from_disk(f'{DATASET_DIR}/train')
val_ds   = load_from_disk(f'{DATASET_DIR}/val')
print(f'Train:{len(train_ds):,}  Val:{len(val_ds):,}')

# Model
processor = WhisperProcessor.from_pretrained(MODEL_ID, language='arabic', task='transcribe')
model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)
model.config.forced_decoder_ids = None
model.config.suppress_tokens    = []
model.config.use_cache          = False
model.generation_config.language = 'arabic'
model.generation_config.task     = 'transcribe'
model.generation_config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language='arabic', task='transcribe')

wer_metric = evaluate.load('wer')

# Collator — on-the-fly feature extraction from Drive
@dataclass
class Collator:
    processor: Any
    decoder_start_token_id: int = 50257
    _err: int = field(default=0, repr=False)

    def __call__(self, features: List[Dict]) -> Dict:
        ins, labs = [], []
        for f in features:
            try:
                a, sr = sf.read(f['audio_path'], dtype='float32', always_2d=False)
                if a.ndim > 1: a = a.mean(axis=1)
                if sr != 16000:
                    import librosa; a = librosa.resample(a, orig_sr=sr, target_sr=16000)
                if len(a) < 1600: raise ValueError('too short')
                feat = self.processor.feature_extractor(a, sampling_rate=16000).input_features[0]
                ids  = self.processor.tokenizer(f['text']).input_ids
            except Exception as e:
                self._err += 1
                if self._err <= 5: print(f'  ⚠️ {f.get("audio_path","?")}: {e}')
                feat = np.zeros((80,3000), dtype='float32')
                ids  = self.processor.tokenizer('').input_ids
            ins.append({'input_features': feat})
            labs.append({'input_ids': ids})

        batch = self.processor.feature_extractor.pad(ins, return_tensors='pt')
        lb = self.processor.tokenizer.pad(labs, return_tensors='pt')
        lbl = lb['input_ids'].masked_fill(lb.attention_mask.ne(1), -100)
        if (lbl[:,0] == self.decoder_start_token_id).all(): lbl = lbl[:,1:]
        batch['labels'] = lbl
        return batch

def compute_metrics(pred):
    p, l = pred.predictions, pred.label_ids
    l[l==-100] = processor.tokenizer.pad_token_id
    ps = processor.tokenizer.batch_decode(p, skip_special_tokens=True)
    ls = processor.tokenizer.batch_decode(l, skip_special_tokens=True)
    return {'wer': round(wer_metric.compute(predictions=ps, references=ls)*100, 4)}

class AdaptiveEval(TrainerCallback):
    def on_evaluate(self, args, state, control, **kw):
        if state.global_step >= 200 and args.eval_steps != 1000:
            args.eval_steps = 1000; print('  eval_steps → 1000')

class MemClean(TrainerCallback):
    def on_step_end(self, args, state, control, **kw):
        if state.global_step % 200 == 0:
            torch.cuda.empty_cache(); gc.collect()

# Training args — auto-detect transformers version
major = int(transformers.__version__.split('.')[0])
kw = dict(
    output_dir=OUTPUT_DIR, per_device_train_batch_size=16,
    per_device_eval_batch_size=8, gradient_accumulation_steps=4,
    learning_rate=1e-5, warmup_steps=500, max_steps=10000,
    gradient_checkpointing=True, fp16=True,
    eval_steps=200, save_strategy='steps', save_steps=200,
    save_total_limit=5, load_best_model_at_end=True,
    metric_for_best_model='wer', greater_is_better=False,
    logging_steps=25, report_to=[], predict_with_generate=True,
    generation_max_length=225, dataloader_num_workers=2,
    dataloader_pin_memory=True, push_to_hub=False,
    remove_unused_columns=False,
)
kw['eval_strategy' if major >= 5 else 'evaluation_strategy'] = 'steps'

tkw = dict(
    model=model, args=Seq2SeqTrainingArguments(**kw),
    train_dataset=train_ds, eval_dataset=val_ds,
    data_collator=Collator(processor=processor,
        decoder_start_token_id=model.config.decoder_start_token_id),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5), AdaptiveEval(), MemClean()],
)
tkw['processing_class' if major >= 5 else 'tokenizer'] = processor.feature_extractor
trainer = Seq2SeqTrainer(**tkw)

# Resume if checkpoint exists
ckpts = sorted(glob.glob(f'{OUTPUT_DIR}/checkpoint-*'))
resume = ckpts[-1] if ckpts else None
print(f'{"Resuming: "+resume if resume else "New training run"}')
print(f'Batch:16×4=64 | Save:every 200 steps | Max:10K steps\n')

trainer.train(resume_from_checkpoint=resume)

FINAL = f'{OUTPUT_DIR}/final'
trainer.save_model(FINAL)
processor.save_pretrained(FINAL)
print(f'\n✅ TRAINING COMPLETE → {FINAL}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 4. Test + Push to HuggingFace
# ══════════════════════════════════════════════════════════════════════════════
import random, torch
from datasets import load_from_disk
from transformers import pipeline
from huggingface_hub import HfApi, login

FINAL = f'{OUTPUT_DIR}/final'
val_ds = load_from_disk(f'{DATASET_DIR}/val')

pipe = pipeline('automatic-speech-recognition', model=FINAL, chunk_length_s=30,
                device=0, generate_kwargs={'language':'arabic','task':'transcribe'})

print('Test results:\n')
for i, s in enumerate(random.sample(list(val_ds), 5), 1):
    try:
        r = pipe(s['audio_path'])
        print(f'[{i}] Expected: {s["text"][:90]}')
        print(f'    Model  : {r["text"][:90]}\n')
    except Exception as e:
        print(f'[{i}] ERROR: {e}\n')

# Push
try:
    from google.colab import userdata
    login(token=userdata.get('HF_TOKEN'))
except Exception:
    print('⚠️ Set HF_TOKEN in Colab secrets to auto-push.')
    print('   Secrets → Add → Key: HF_TOKEN → Value: your token from huggingface.co/settings/tokens')
    login()

card = f"""---
language: ar
license: mit
tags: [whisper, quran, arabic, speech-recognition, fine-tuned]
base_model: openai/whisper-small
pipeline_tag: automatic-speech-recognition
---
# QuranASR — Whisper Fine-tuned for Quran Recitation

```python
from transformers import pipeline
pipe = pipeline('automatic-speech-recognition', model='{HF_REPO_ID}',
                generate_kwargs={{'language':'arabic','task':'transcribe'}})
print(pipe('quran.mp3')['text'])
```

12 reciters × 6236 verses | batch 64 | lr 1e-5 | A100

[GitHub](https://github.com/tialkan/QuranASR) · [Tilavet App](https://tilavet.app) · [Author](https://huggingface.co/baristiran)
"""
with open(f'{FINAL}/README.md','w') as f: f.write(card)

api = HfApi()
api.create_repo(repo_id=HF_REPO_ID, exist_ok=True)
api.upload_folder(folder_path=FINAL, repo_id=HF_REPO_ID,
                  commit_message='Upload QuranASR whisper-small')
print(f'\n✅ Live at: https://huggingface.co/{HF_REPO_ID}')